## Data Quality

O código apresentado neste notebook foi desenvolvido durante a análise da qualidade da base de dados e tem como objetivo compreender possíveis regras de negócio e como elas estão representadas na base de dados, além de identificar inconsistências, problemas de tipagem, valores ausentes, duplicatas e outros possíveis problemas de qualidade.

In [61]:
import os
import pandas as pd

from pathlib import Path

from dotenv import load_dotenv

In [62]:
load_dotenv()

True

In [63]:
dataset_path = os.getenv("DATASET_PATH")

df = pd.read_csv(dataset_path, sep="|", low_memory=False)

## Seleção de variáveis

Em um primeiro momento faço uma investigação apenas nas colunas que parecem alinhadas com nosso objetivo de pesquisa: uma análise de justiça algoritmica de modelos de machine learning no cenário de microcrédito. Essas colunas fazem parte dos grupos:

1. Dados da proposta
2. Dados dos grupos de pessoas
3. Dados de pesquisas cadastrais
4. Dados do micronegócio
5. Dados dos avalistas
6. Indicadores financeiros do micronegócio
7. Dados de emprestimos individuais realizados pelos donos do micronegócio

In [64]:
COLUNA_IDENTIFICADORA = [
    "chave"
]

COLUNAS_DADOS_PROPOSTA = [
    "DE_PRD_PRP",
    "PR_APR_PRP",
    "VR_PRZ_APR",
    "QT_PRS_APR",
    "MT_ASS_CRD",
    "TX_JRS_APR",
    "VR_APR_PRP",
    "VR_SOL_PRP",
    "VR_JRS_APR",
    "VR_SEG_PRP",
    "DT_CAD_PRP",
    "VR_PRS_PRP",
    "VR_MNR_PRS",
    "VR_MAI_PRS",
    "VR_MED_PRS",
    "ID_TP_PRP",
    "QT_DD_ACM_ATR",
    "QT_DD_ACM_ATR_MAX"
]

COLUNAS_DADOS_GRUPOS_PESSOAS = [
    "QT_CLI",
    "VR_PC_CLI_NOV",
    "PERC_CLI_NOV_GRP",
    "VR_IDD_MNM",
    "VR_IDD_MAX",
    "VR_IDD_MED",
    "QT_CLI_ANF",
    "QT_CLI_PRI_GR",
    "QT_CLI_SEG_GR",
    "QT_CLI_SUP",
    "QT_CLI_STR",
    "QT_CLI_CAS",
    "QT_CLI_SPD",
    "QT_RES_PRP",
    "QT_RES_ALU",
    "QT_RES_OUT",
    "QT_CLI_MLH",
    "QT_CLI_HOM",
    "QT_RES_IGL",
    "QT_ZON_IGL",
    "DE_ZON_FRQ",
    "QT_CJG_CLI",
    "QT_REN_CLI",
    "VR_PC_DPR"
]

COLUNAS_DADOS_PESQUISAS_CADASTRAIS = [
    "QT_NEG_SPC",
    "QT_NEG_SPC_MED",
    "QT_NEG_SERASA",
    "QT_NEG_SERASA_MED",
    "QT_NEG_BNB",
    "QT_NEG_BNB_MED"
]

COLUNAS_DADOS_MICRONEGOCIO = [
    "QT_NEG_FIX",
    "QT_NEG_AMB",
    "QT_STR_BRC",
    "QT_STR_MVL",
    "QT_STR_PNT",
    "QT_STR_DOM",
    "QT_STR_CSA",
    "QT_NIV_SBS",
    "QT_NIV_SMP",
    "QT_NIV_AMP",
    "QT_NIV_POR",
    "QT_ATV_CFC",
    "QT_ATV_CRL",
    "QT_ATV_BEB",
    "QT_ATV_ANI",
    "QT_ATV_PFM",
    "QT_ATV_OUT",
    "DE_ATV_FRQ",
    "QT_FAM_TOT",
    "QT_NAO_TOT",
    "QT_RES_TOT",
    "QT_CPR_VRJ",
    "QT_CPR_ATC",
    "QT_CPR_FAB",
    "QT_CPR_PRD",
    "QT_SET_IND",
    "QT_SET_COM",
    "QT_SET_SER",
    "VR_DSV_TPO",
    "QT_OUT_NEG",
    "QT_EMP_FML",
    "QT_MUN_NEG",
    "QT_RTT_DIA",
    "QT_RTT_QZN",
    "QT_RTT_MEN",
    "QT_RTT_SMN",
    "QT_PRZ_AVT",
    "QT_PRZ_15",
    "QT_PRZ_30",
    "QT_PRZ_60",
    "QT_CTL_NAO",
    "QT_CTL_PCR",
    "QT_CTL_STF",
    "QT_CTL_BOM",
    "QT_FIX_PRP",
    "QT_FIX_ALU",
    "FLG_ATV"
]

COLUNAS_DADOS_AVALISTAS = [
    "QT_AVL",
    "QT_AVL_EPR_ABE",
    "QT_TOT_EPR_FCH",
    "VR_TOT_EPR_FCH",
    "VR_MED_EPR_FCH"
]

COLUNAS_OUTRAS = [
    "QT_CLI_SPC",
    "QT_CLI_SRS",
    "QT_CLI_BNB",
    "QT_TPO_ULT_BNB_MNM",
    "QT_TPO_ULT_BNB_MED",
    "QT_TPO_ULT_BNB_MAX"
]

COLUNAS_INDICADORES_FINANCEIROS_MICRONEGOCIO = [
    "VR_CLO_OPR_MED",
    "VR_NEC_INV_MED",
    "VR_LIQ_COR_MED",
    "VR_MRG_LCR_OPR_MED",
    "VR_EDV_MED"
]

COLUNAS_EMPRESTIMOS_INDIVIDUAIS = [
    "QT_CLI_EPR_IND_PGO",
    "QT_CLI_EPR_IND_ABE",
    "QT_CLI_IND_ABE_GRP",
    "VR_ULT_EPR_IND_MED",
    "QT_PCL_ULT_EPR_MED"
]

In [65]:
colunas_selecionadas = (
    COLUNA_IDENTIFICADORA +
    COLUNAS_DADOS_PROPOSTA +
    COLUNAS_DADOS_GRUPOS_PESSOAS +
    COLUNAS_DADOS_PESQUISAS_CADASTRAIS +
    COLUNAS_DADOS_MICRONEGOCIO +
    COLUNAS_DADOS_AVALISTAS +
    COLUNAS_OUTRAS +
    COLUNAS_INDICADORES_FINANCEIROS_MICRONEGOCIO +
    COLUNAS_EMPRESTIMOS_INDIVIDUAIS
)

df_filtrado = df[colunas_selecionadas]

## Limpeza de dados

Inicialmente realizo uma análise dos dados ausentes na minha base de dados.

In [66]:
ausentes = pd.DataFrame({
    "total de ausentes": df_filtrado.isna().sum(),
    "percentual de ausentes": df_filtrado.isna().mean() * 100
})

ausentes = ausentes.sort_values(
    "percentual de ausentes",
    ascending=False
)

ausentes

,total de ausentes,percentual de ausentes
DE_ZON_FRQ,640489,100.000000
QT_ZON_IGL,640489,100.000000
QT_TPO_ULT_BNB_MED,607583,94.862363
QT_TPO_ULT_BNB_MNM,607583,94.862363
QT_TPO_ULT_BNB_MAX,607583,94.862363
...,...,...
QT_PRZ_60,0,0.000000
QT_CTL_NAO,0,0.000000
QT_CTL_PCR,0,0.000000
QT_CTL_STF,0,0.000000


Como as colunas DE_ZON_FRQ e QT_ZON_IGL estão totalmente vazias, elas devem ser retiradas da base de dados.

As colunas com **informações dos avalistas** do empréstimo possuem cerca de 83 à 91% de ausência de dados. É importante confirmarmos se todos os tipos de produtos (DE_PRD_PRP) e todos os tipos de propostas (ID_TP_PRP) necessitam de avalistas para termos certeza se isso é uma falha de registro ou uma regra de negócio.

Além disso, informações a respeito dos **empréstimos individuais dos clientes** também encontram-se ausentes em entre 63 à 81% da base de dados. Também é interessante verificar em que situações estes dados demonstram ser importantes e necessários para confirmarmos a hipótese de falha de registro.

### Análise dos dados sobre avalistas

- Análise por tipo de proposta

In [67]:
avalistas_por_proposta = df_filtrado.groupby("ID_TP_PRP")["QT_AVL"].agg(
    total="size",
    missing="count"
)

avalistas_por_proposta["missing"] = avalistas_por_proposta["total"] - avalistas_por_proposta["missing"]
avalistas_por_proposta["missing_percentage"] = avalistas_por_proposta["missing"]/avalistas_por_proposta["total"] * 100
avalistas_por_proposta

,total,missing,missing_percentage
ID_TP_PRP,,,
0,485682,394295,81.183779
1,145183,130549,89.920307
2,9624,8032,83.458022


- Análise por tipo de produto

In [68]:
avalistas_por_produto = df_filtrado.groupby("DE_PRD_PRP")["QT_AVL"].agg(
    total="size",
    missing="count"
)

avalistas_por_produto["missing"] = avalistas_por_produto["total"] - avalistas_por_produto["missing"]
avalistas_por_produto["missing_percentage"] = avalistas_por_produto["missing"]/avalistas_por_produto["total"] * 100
avalistas_por_produto

,total,missing,missing_percentage
DE_PRD_PRP,,,
1,324430,324430,100.000000
2,51108,743,1.453784
3,57665,1076,1.865950
4,199606,199606,100.000000
5,7010,7010,100.000000
7,670,11,1.641791


QT_AVL apresenta padrão de ausência fortemente associado ao tipo de produto (DE_PRD_PRP). Nos produtos 1, 4 e 5, a variável apresenta ausência total, enquanto nos produtos 2, 3 e 7 a ausência é inferior a 2%. Esse padrão sugere que a ausência pode estar relacionada à aplicabilidade da informação (necessidade de se ter avalista) conforme a modalidade de crédito, e não necessariamente representar falha de preenchimento.

Supoẽ-se que o produto 1 é o sistema Crediamigo. Este produto de crédito é feito em grupo e se assenta no conceito de "Aval Solidário", a exigência da reunião de um grupo de empreendedores locais/amigos de forma que a responsabilidade pelo serviço da dívida é coletiva; se um membro do grupo entrar em incumprimento, os restantes assumem o ónus financeiro, sendo portanto não necessário avalista.

### Análise dos dados sobre empréstimo individual

- Análise por tipo de proposta

In [69]:
emp_indv_por_proposta = df_filtrado.groupby("ID_TP_PRP")["QT_CLI_EPR_IND_PGO"].agg(
    total="size",
    missing="count"
)

emp_indv_por_proposta["missing"] = emp_indv_por_proposta["total"] - emp_indv_por_proposta["missing"]
emp_indv_por_proposta["missing_percentage"] = emp_indv_por_proposta["missing"]/emp_indv_por_proposta["total"] * 100
emp_indv_por_proposta

,total,missing,missing_percentage
ID_TP_PRP,,,
0,485682,327772,67.486956
1,145183,120007,82.659127
2,9624,7517,78.106816


- Análise por tipo de produto

In [70]:
emp_indv_por_produto = df_filtrado.groupby("DE_PRD_PRP")["QT_CLI_EPR_IND_PGO"].agg(
    total="size",
    missing="count"
)

emp_indv_por_produto["missing"] = emp_indv_por_produto["total"] - emp_indv_por_produto["missing"]
emp_indv_por_produto["missing_percentage"] = emp_indv_por_produto["missing"]/emp_indv_por_produto["total"] * 100
emp_indv_por_produto

,total,missing,missing_percentage
DE_PRD_PRP,,,
1,324430,239137,73.709891
2,51108,24492,47.922047
3,57665,27957,48.481748
4,199606,158443,79.377874
5,7010,4913,70.085592
7,670,354,52.835821


Nessa análise, observamos que a ausência de dados a respeito da emprestimos individuais realizados pelo cliente possuem uma consistencia de nulidade entre os diferentes tipos de proposta e os diferentes tipos de produtos. Dessa forma, podemos optar por descartar esses dados sob o pretexto de ter ocorrido falha de registro.

### Remoção de dados analisados

Para facilitar a análise das próximas colunas, vou remover as que analisamos agora e não se demonstraram pertinentes e salvar o dataframe em um novo csv

In [71]:
COLUNAS_DADOS_AVALISTAS = [
    "QT_AVL",
    "QT_AVL_EPR_ABE",
    "QT_TOT_EPR_FCH",
    "VR_TOT_EPR_FCH",
    "VR_MED_EPR_FCH"
]

COLUNAS_EMPRESTIMOS_INDIVIDUAIS = [
    "QT_CLI_EPR_IND_PGO",
    "QT_CLI_EPR_IND_ABE",
    "QT_CLI_IND_ABE_GRP",
    "VR_ULT_EPR_IND_MED",
    "QT_PCL_ULT_EPR_MED"
]

OUTRAS_COLUNAS = [
    "DE_ZON_FRQ", 
    "QT_ZON_IGL"
]

In [72]:
colunas_para_remover = (COLUNAS_DADOS_AVALISTAS + COLUNAS_EMPRESTIMOS_INDIVIDUAIS + OUTRAS_COLUNAS)

df_filtrado.drop(colunas_para_remover, axis=1, inplace=True)

Seleciono apenas os dados que são referentes ao produto 1 pois suponho que são os dados do crediamigo.

In [73]:
df_output = df_filtrado[df_filtrado['DE_PRD_PRP'] == 1]

In [77]:
df_output.drop(columns=["QT_TPO_ULT_BNB_MNM", "QT_TPO_ULT_BNB_MED", "QT_TPO_ULT_BNB_MAX"], inplace=True)

In [78]:
ausentes = pd.DataFrame({
    "total de ausentes": df_output.isna().sum(),
    "percentual de ausentes": df_output.isna().mean() * 100
})

ausentes = ausentes.sort_values(
    "percentual de ausentes",
    ascending=False
)

ausentes.head(10)

,total de ausentes,percentual de ausentes
DE_ATV_FRQ,178035,54.876244
QT_CLI_BNB,8357,2.575902
VR_NEC_INV_MED,8357,2.575902
VR_LIQ_COR_MED,8357,2.575902
VR_MRG_LCR_OPR_MED,8357,2.575902
VR_CLO_OPR_MED,8357,2.575902
QT_CLI_SPC,8357,2.575902
QT_CLI_SRS,8357,2.575902
VR_EDV_MED,8357,2.575902
QT_NEG_BNB_MED,8356,2.575594


No dataset há 324 mil registros, porém em geral as colunas possuem cerca de 2 à 3% de valores ausentes. Com exceção das colunas **QT_TPO_ULT_BNB_MNM, QT_TPO_ULT_BNB_MED e QT_TPO_ULT_BNB_MAX** que representam respectivamente: o mínimo, a média e o máximo de tempo desde à última passagem negativa pelo BNB, que possuem 94% dos valores ausentes. Devido a isso, essas colunas serão removidas.

A coluna **indicadora de atividade mais frequente do micronegócio** apesar de possuir cerca de 54% dos valores nulos possui um valor relevante para a análise dos nossos dados. Dessa forma, estou optando por remover de todo meu dataset todas as linhas que possuem dados faltantes com o objetivo de facilitar o tratamento final dos dados mas ainda mantendo os registros que nos trazem informação. 

In [84]:
df_output.dropna(inplace=True)
df_output.info()

<class 'pandas.DataFrame'>
Index: 143168 entries, 8 to 640487
Columns: 102 entries, chave to VR_EDV_MED
dtypes: float64(29), int64(68), str(5)
memory usage: 112.5 MB


Ficamos com cerca de 143 mil registros, o que é 140x maior que o German Credit Card, principal dataset que os pesquisadores utilizam nos artigos sobre justiça algorítmica aplicada a crédito. Dessa forma, acho que essa ação (de remover dados nulos) não será prejudicial.

In [ ]:
filepath = Path("../dataset/crediamigo.csv")
df_output.to_csv(filepath)